# Chapter 11 &mdash; Disambiguation by Layering: Expression, Term, Factor

**Concept 10 of the Chapter 11 decomposition:** *Disambiguation by Layering: Expression, Term, Factor*

Force precedence by making a higher layer call a lower one &mdash; same language, one tree.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Disambiguation-By-Layering/Concept-Disambiguation-By-Layering.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The standard cure for expression ambiguity is **layering**, one layer per precedence
level:

```
E -> T | E+T          expression: additions, left-associating
T -> F | T*F          term:       multiplications
F -> 1|2|3|~F|(E)     factor:     atoms and parentheses
```

Two mechanisms are at work:

* **precedence** &mdash; `+` can only combine *terms*, so a `*` is always bound tighter;
* **associativity** &mdash; recursion on the **left** only (`E+T`, not `E+E`) forces
  left-association.

The language is unchanged; only the tree shape is. That is the point: same strings,
one tree each.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The two grammars

In [ ]:
Amb   = mkg({'E': ["1", "2", "3", "~E", "E+E", "E*E", "(E)"]}, 'E')
Layer = mkg({'E': ["T", "E+T"],
             'T': ["F", "T*F"],
             'F': ["1", "2", "3", "~F", "(E)"]}, 'E')
show(Layer)

### Evaluation, so structure can be read as a number

In [ ]:
def ev(t):
    if isinstance(t, str): return int(t)
    kids = t[1:]
    if len(kids) == 1: return ev(kids[0])
    if len(kids) == 2 and kids[0] == '~': return -ev(kids[1])
    if len(kids) == 3 and kids[0] == '(': return ev(kids[1])
    a, op, b = kids
    return ev(a) + ev(b) if op == '+' else ev(a) * ev(b)

## 3. Tests

**Same language.** Layering changes the trees, not the strings.

In [ ]:
for n in [4, 5, 6]:
    a, b = set(language(Amb, n)), set(language(Layer, n))
    print("up to length %d : %d strings each, equal? %s" % (n, len(a), a == b))
    assert a == b

**One tree each.** Where the ambiguous grammar had 2 or 5, the layered one has 1.

In [ ]:
print("%-12s %-10s %-10s" % ("string", "ambiguous", "layered"))
for w in ['1', '1+2', '1+2*3', '1+2*3+1', '1+2*3+1*2', '~1+2']:
    print("%-12s %-10d %-10d" % (w, nparses(Amb, w), nparses(Layer, w)))
    assert nparses(Layer, w) == 1

And the single tree gives the **conventional** answer.

In [ ]:
for w in ['1+2*3', '1*2+3', '1+2+3', '1+2*3+1']:
    v = ev(parse_trees(Layer, w, cap=1)[0])
    print("  %-10s evaluates to %-3d  (Python says %d)" % (w, v, eval(w)))
    assert v == eval(w)

**Precedence** comes from `+` combining only *terms*.

In [ ]:
t = parse_trees(Layer, '1+2*3', cap=1)[0]
show_tree(t)
print("\nthe * sits under a T, which sits under the right operand of + --")
print("so * is evaluated first, structurally.")

**Associativity** comes from recursing on the left only.

In [ ]:
t = parse_trees(Layer, '1+2+3', cap=1)[0]
show_tree(t)
print("\nE+T with E recursing left => (1+2)+3, left-associative.")
Right = mkg({'E': ["T", "T+E"], 'T': ["F", "F*T"], 'F': ["1","2","3","(E)"]}, 'E')
print("\nright-recursive version, 1+2+3 :")
show_tree(parse_trees(Right, '1+2+3', cap=1)[0])
assert nparses(Right, '1+2+3') == 1
print("\nSame language again -- but 1+(2+3).  For + it does not matter; for - it would.")

## 4. Exercises


1. Add unary minus at the right precedence. Which layer does it belong to?
2. Make `*` right-associative. Which production changes?
3. Add a `^` (power) layer that binds tighter than `*` and associates right.

In [ ]:
# Your work for the exercises above.